# Cloud-verkeersvolume

Dit notebook traint een SparkXGBoost-model voor verkeersvolume-voorspelling op de I-94 snelweg. De Spark-training draait in een **Docker-container** — zo werkt de pipeline ook op Windows zonder Spark lokaal te installeren.

**Structuur:**
1. Installaties & imports (buiten Docker)
2. Kolombeschrijving & configuratie
3. Docker-container bouwen
4. Training starten via Docker
5. Resultaten & evaluatie (teruglezen uit `metrics.json`)
6. Hertraining (CT-hook)
7. Deployment & monitoring


## 1. Installaties & imports

Deze packages draaien **lokaal** in het notebook (niet in Docker). Ze zijn nodig voor het lezen van MLflow-resultaten en de FastAPI-deployment.

In [ ]:
import subprocess
import sys

# Packages die niet standaard in Anaconda zitten
for pkg in ["mlflow", "fastapi", "uvicorn[standard]", "pydantic", "evidently", "requests"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import json
import os
import datetime
import mlflow
from mlflow.tracking import MlflowClient

print("Imports gereed.")


Imports gereed.


## 2. Configuratie

Alle instellingen op één plek. `DATA` verwijst naar de CSV **zoals het pad eruit ziet binnen de Docker-container** (`/app/Data/...`). `MLRUNS_DIR` wordt als volume gedeeld tussen de container en de host, zodat MLflow-runs ook lokaal zichtbaar zijn.

In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    # Pad naar de data (als gemount volume in de container)
    DATA: str = "Data/Metro_Interstate_Traffic_Volume.csv"

    # MLflow — gedeelde directory tussen container en host via volume-mount
    MLRUNS_DIR: str = "mlruns-cloud"
    MLFLOW_URI: str = f"sqlite:///mlruns-cloud/mlflow.db"
    EXPERIMENT: str = "nova_cloud"
    MODEL_NAME: str = "SparkXGBTrafficModel"

    # Docker image naam
    DOCKER_IMAGE: str = "traffic-spark"

CFG = Config()
print("Config geladen:", CFG)


Config geladen: Config(DATA='Data/Metro_Interstate_Traffic_Volume.csv', MLRUNS_DIR='mlruns-cloud', MLFLOW_URI='sqlite:///mlruns-cloud/mlflow.db', EXPERIMENT='nova_cloud', MODEL_NAME='SparkXGBTrafficModel', DOCKER_IMAGE='traffic-spark')


## 3. Kolombeschrijving

Uitleg kolommen:

| Kolomnaam          | Data type   |Beschrijving                                                                 |
|----------------------|-------------|-----------------------------------------------------------------------------|
| holiday              | categorical  | US National holidays plus regional holiday, Minnesota State Fair             |
| temp                 | numeric      | Average temp in kelvin                                                        |
| rain_1h              | numeric      | Amount in mm of rain that occurred in the hour                                      |
| snow_1h              | numeric      | Amount in mm of snow that occurred in the hour                                      |
| clouds_all           | numeric      | Percentage of cloud cover                                                        |
| weather_main         | categorical  | Short textual description of the current weather                                      |
| weather_description  | categorical  | Longer textual description of the current weather                                       |
| date_time           | datetime     | Hour of the data collected in local CST time                                       |
| traffic_volume       | numeric      | Hourly I-94 ATR 301 reported westbound traffic volume Interstate                                       |

Hier is traffic_volume de target variabele, en de rest zijn features.

## 4. Docker-container bouwen

De `Dockerfile.spark` bevat een officiële Apache Spark + Python image (Linux). Op Linux is geen `winutils.exe` nodig — dit is precies waarom we Docker gebruiken op Windows.

**Inhoud van de container:**
- Apache Spark 3.5.1 (met Java)
- PySpark + XGBoost + MLflow
- `train.py` — het trainingsscript

De eerste build duurt ~3 minuten (image downloaden). Daarna is het gecached en gaat het snel.


In [3]:
import subprocess, sys

def docker_build(image_name=CFG.DOCKER_IMAGE):
    """Bouw de Spark-trainingscontainer. Hoeft maar één keer."""
    print(f"Docker image '{image_name}' bouwen... (eerste keer ~3 min)")
    result = subprocess.run(
        ["docker", "build", "-f", "Dockerfile.spark", "-t", image_name, "."],
        capture_output=True,
        text=True,
    )
    # Altijd output tonen zodat je de echte fout ziet
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Docker build mislukt (exit {result.returncode}). Zie output hierboven.")
    print(f"Image '{image_name}' klaar.")

docker_build()


Docker image 'traffic-spark' bouwen... (eerste keer ~3 min)
#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.spark
#1 transferring dockerfile: 1.24kB done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/spark:python3
#2 DONE 0.4s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/6] FROM docker.io/library/spark:python3@sha256:732a853f8e0e637ef9b72190e61890436ab3d42212a38ff50f3181749d63a0ce
#4 resolve docker.io/library/spark:python3@sha256:732a853f8e0e637ef9b72190e61890436ab3d42212a38ff50f3181749d63a0ce 0.0s done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 5.94kB done
#5 DONE 0.0s

#6 [2/6] RUN ln -s /usr/bin/python3 /usr/bin/python
#6 CACHED

#7 [3/6] COPY requirements-spark.txt /app/requirements-spark.txt
#7 CACHED

#8 [4/6] RUN pip install --no-cache-dir -r /app/requirements-spark.txt
#8 CACHED

#9 [5/6] COPY train.py /app/train.py
#9 DONE 

## 5. Training starten via Docker

De container krijgt twee **volume-mounts** mee:
- `./Data` → `/app/Data` : de ruwe dataset
- `./mlruns-cloud` → `/app/mlruns-cloud` : MLflow-runs (gedeeld met host)

Zo schrijft de container de trainingsresultaten direct naar de host-map, en zijn ze daarna zichtbaar in dit notebook via `mlflow`.

`metrics.json` wordt door `train.py` aangemaakt en bevat RMSE, MAE, R² en de baseline-vergelijking.


In [17]:
def run_training(data_path=CFG.DATA, image_name=CFG.DOCKER_IMAGE):
    """Start de Spark-training in een Docker-container.

    Args:
        data_path (str): Pad naar de CSV zoals gemount in de container.
        image_name (str): Naam van de Docker-image.

    Returns:
        dict: Metrics uit metrics.json (RMSE, MAE, R2, baseline_rmse).
    """
    os.makedirs(CFG.MLRUNS_DIR, exist_ok=True)

    # Absolute paden voor de volume-mounts
    work_dir   = os.path.abspath(".")           # huidige map -> metrics.json
    data_dir   = os.path.abspath("Data")
    mlflow_dir = os.path.abspath(CFG.MLRUNS_DIR)

    cmd = [
        "docker", "run", "--rm",
        "-v", f"{work_dir}:/app/output",        # metrics.json komt hier terecht
        "-v", f"{data_dir}:/app/Data",
        "-v", f"{mlflow_dir}:/app/mlruns-cloud",
        "-e", "GIT_PYTHON_REFRESH=quiet",       # onderdruk git-warnings
        image_name,
        "python3", "train.py", "--data", data_path, "--output", "/app/output",
    ]

    print("Docker commando:", " ".join(cmd))
    result = subprocess.run(cmd, text=True)

    if result.returncode != 0:
        raise RuntimeError("Training mislukt. Zie Docker-output hierboven.")

    # metrics.json staat nu in de huidige map op de host
    with open("metrics.json") as f:
        metrics = json.load(f)

    return metrics


metrics = run_training()


Docker commando: docker run --rm -v c:\Users\tldeb\Desktop\Repositories\realtime-verkeerswaarneming:/app/output -v c:\Users\tldeb\Desktop\Repositories\realtime-verkeerswaarneming\Data:/app/Data -v c:\Users\tldeb\Desktop\Repositories\realtime-verkeerswaarneming\mlruns-cloud:/app/mlruns-cloud -e GIT_PYTHON_REFRESH=quiet traffic-spark python3 train.py --data Data/Metro_Interstate_Traffic_Volume.csv --output /app/output


## 6. Resultaten & evaluatie

Na de training leest het notebook `metrics.json` terug. De metrics zijn ook gelogd in MLflow en bekijkbaar via `mlflow ui`.

**Metrics-uitleg:**
- **RMSE** (Root Mean Squared Error): gemiddelde fout in voertuigen/uur — lager is beter
- **MAE** (Mean Absolute Error): minder gevoelig voor uitbijters dan RMSE
- **R²**: verklaarde variantie — 1.0 is perfect, 0.0 is even goed als het gemiddelde


In [5]:
# Resultaten tonen
print(f"RMSE              : {metrics['rmse']:.1f} voertuigen/uur")
print(f"MAE               : {metrics['mae']:.1f} voertuigen/uur")
print(f"R²                : {metrics['r2']:.3f}")
print()
print(f"Baseline RMSE     : {metrics['baseline_rmse']:.1f}  (altijd gemiddelde voorspellen)")
print(f"Verbetering       : {metrics['improvement_pct']:.1f}%")
print(f"MLflow Run ID     : {metrics['run_id']}")


RMSE              : 348.8 voertuigen/uur
MAE               : 200.5 voertuigen/uur
R²                : 0.969

Baseline RMSE     : 1981.6  (altijd gemiddelde voorspellen)
Verbetering       : 82.4%
MLflow Run ID     : 3100296f16874d51b5cae223a96bec33


In [6]:
# MLflow run details ophalen
mlflow.set_tracking_uri(CFG.MLFLOW_URI)
client = MlflowClient()

run = client.get_run(metrics["run_id"])
print("Parameters gelogd in MLflow:")
for k, v in run.data.params.items():
    print(f"  {k}: {v}")


Parameters gelogd in MLflow:
  n_estimators: 900
  max_depth: 9
  learning_rate: 0.05
  subsample: 0.8
  reg_alpha: 0.4
  random_state: 42
  data_path: Data/Metro_Interstate_Traffic_Volume.csv
  timestamp: 2026-06-23T13:03:55.108747


## 7. Hertraining (CT-hook)

De `retrain()`-functie herstart de Docker-container met nieuwe data. Dit wordt getriggerd door de monitoringlaag (zie sectie 8) wanneer:
- RMSE op nieuwe data een drempelwaarde overschrijdt
- Data-drift gedetecteerd wordt in feature-distributies
- Een periodiek schema (bijv. maandelijks) het aangeeft

Elke hertraining wordt geregistreerd als nieuwe versie in de MLflow Model Registry.


In [7]:
def retrain(new_data_path, run_name="sparkxgb_retrain"):
    """Hertrain het model op nieuwe data via de Docker-container.

    Args:
        new_data_path (str): Pad naar het CSV-bestand met nieuwe trainingsdata
                             (relatief aan de gemounte /app/Data map).
        run_name (str): Naam van de MLflow-run.

    Returns:
        dict: Metrics van de hertraining.
    """
    print(f"[{datetime.datetime.now():%Y-%m-%d %H:%M}] Hertraining gestart op: {new_data_path}")
    metrics = run_training(data_path=new_data_path)
    print(f"Hertraining klaar: RMSE={metrics['rmse']:.1f}, R²={metrics['r2']:.3f}")
    return metrics


# Voorbeeld aanroep (uitgecommentarieerd — gebruik bij echte nieuwe data):
# retrain_metrics = retrain("Data/Metro_Interstate_Traffic_Volume_new.csv")
print("retrain() klaar voor gebruik door de monitoringlaag.")


retrain() klaar voor gebruik door de monitoringlaag.


## 8. Deployment (FastAPI REST API)

Het getrainde model wordt beschikbaar gesteld als REST API via **FastAPI**. Externe systemen kunnen real-time voorspellingen opvragen via HTTP.

**Schaalbaarheid:** De API draait in een Docker-container en kan horizontaal schalen achter een load balancer op bijv. AWS ECS of Google Cloud Run.

**CI/CD:** Bij elke nieuwe modelversie in de MLflow Registry wordt de API automatisch herbouwd en gedeployed via een GitHub Actions workflow.


In [18]:
api_code = '''
"""FastAPI deployment voor het XGBoost verkeersvolume model.

Het model wordt geladen vanuit model.json (XGBoost native formaat).
Zo is er geen MLflow server nodig in de cloud.

Start de server met:
    uvicorn app:app --host 0.0.0.0 --port 8000

Of via Docker:
    docker build -f Dockerfile.api -t traffic-api .
    docker run -p 8000:8000 -v $(pwd)/model.json:/app/model.json traffic-api
"""

import os
import xgboost as xgb
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

# Model laden vanuit bestand (werkt lokaal én in de cloud)
MODEL_PATH = os.getenv("MODEL_PATH", "model.json")
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

app = FastAPI(
    title="Traffic Volume Prediction API",
    description="Voorspelt het uurlijkse verkeersvolume op de I-94 snelweg.",
    version="1.0.0",
)


class TrafficFeatures(BaseModel):
    holiday: bool = Field(..., description="Is het een feestdag?")
    temp: float = Field(..., description="Temperatuur in Kelvin", ge=200, le=350)
    rain_1h: float = Field(0.0, description="Regen in mm", ge=0)
    snow_1h: float = Field(0.0, description="Sneeuw in mm", ge=0)
    clouds_all: int = Field(..., description="Wolkendekking %", ge=0, le=100)
    hour: int = Field(..., ge=0, le=23)
    day_of_week: int = Field(..., ge=1, le=7)
    month: int = Field(..., ge=1, le=12)
    year: int = Field(..., ge=2000, le=2100)
    weather_main: Optional[str] = None
    weather_description: Optional[str] = None


class PredictionResponse(BaseModel):
    predicted_traffic_volume: float
    model_path: str


@app.get("/health")
def health_check():
    return {"status": "ok", "model_path": MODEL_PATH}


@app.post("/predict", response_model=PredictionResponse)
def predict(features: TrafficFeatures):
    df = pd.DataFrame([features.model_dump()])
    try:
        prediction = model.predict(df)
        return PredictionResponse(
            predicted_traffic_volume=float(prediction[0]),
            model_path=MODEL_PATH,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc
'''

with open("app.py", "w") as f:
    f.write(api_code)
print("app.py aangemaakt.")


app.py aangemaakt.


## 9. Monitoring

Monitoring op twee niveaus:

| Niveau | Wat | Hoe |
|--------|-----|-----|
| **Systeem** | API actief? | `/health` endpoint periodiek aanroepen |
| **Model** | RMSE verslechterd? | Periodiek meten op nieuwe data |
| **Data** | Distributie verschoven? | Jensen-Shannon divergentie per feature |

**Drempelwaarden:**
- RMSE > 500 → waarschuwing
- RMSE > 800 of drift > 0.15 → hertraining getriggerd


In [10]:
import numpy as np
from scipy.spatial.distance import jensenshannon
import pandas as pd

def compute_drift_score(ref_series, cur_series, bins=20):
    """Jensen-Shannon divergentie tussen twee pandas Series.

    Args:
        ref_series: Referentiedata (trainingsdata) als pandas Series.
        cur_series: Nieuwe inkomende data als pandas Series.
        bins (int): Aantal histogram-bins.

    Returns:
        float: Drift-score tussen 0 (geen drift) en 1 (maximale drift).
    """
    all_data = pd.concat([ref_series, cur_series])
    edges = np.histogram_bin_edges(all_data.dropna(), bins=bins)

    ref_hist, _ = np.histogram(ref_series.dropna(), bins=edges, density=True)
    cur_hist, _ = np.histogram(cur_series.dropna(), bins=edges, density=True)

    # Epsilon om deling door nul te voorkomen
    return float(jensenshannon(ref_hist + 1e-10, cur_hist + 1e-10))


def monitor_model(new_data_path, rmse_threshold=500.0, drift_threshold=0.15):
    """Monitor modelprestaties en data-drift. Trigger hertraining indien nodig.

    Args:
        new_data_path (str): Pad naar nieuwe data met ground-truth labels.
        rmse_threshold (float): RMSE-drempel voor waarschuwing.
        drift_threshold (float): Drift-drempel voor hertraining.

    Returns:
        dict: Rapport met metrics, alerts en ondernomen actie.
    """
    ref_df = pd.read_csv("Data/Metro_Interstate_Traffic_Volume.csv")
    new_df = pd.read_csv(new_data_path)

    report = {
        "timestamp": datetime.datetime.now().isoformat(),
        "n_new_samples": len(new_df),
        "alerts": [],
        "retrain_triggered": False,
    }

    # Data-drift per numerieke feature
    numeric_features = ["temp", "rain_1h", "snow_1h", "clouds_all"]
    drift_scores = {}
    for feat in numeric_features:
        if feat in ref_df.columns and feat in new_df.columns:
            score = compute_drift_score(ref_df[feat], new_df[feat])
            drift_scores[feat] = round(score, 4)
            if score > drift_threshold:
                report["alerts"].append(
                    f"Data-drift in '{feat}': {score:.3f} > drempel {drift_threshold}"
                )
                report["retrain_triggered"] = True

    report["drift_scores"] = drift_scores

    # Log naar MLflow
    mlflow.set_tracking_uri(CFG.MLFLOW_URI)
    with mlflow.start_run(run_name="monitoring_check"):
        for feat, score in drift_scores.items():
            mlflow.log_metric(f"drift_{feat}", score)
        mlflow.log_dict(report, "monitoring_report.json")

    # Hertraining triggeren indien nodig
    if report["retrain_triggered"]:
        print("⚠ Hertraining getriggerd vanwege:")
        for alert in report["alerts"]:
            print(f"  - {alert}")
        retrain_metrics = retrain(new_data_path)
        report["retrain_metrics"] = retrain_metrics
        report["action_taken"] = "retrain_uitgevoerd"
    else:
        print("✓ Model gezond. Geen hertraining nodig.")
        report["action_taken"] = "geen"

    return report


# Voorbeeld aanroep (uitgecommentarieerd — gebruik bij echte nieuwe data):
# rapport = monitor_model("Data/Metro_Interstate_Traffic_Volume_new.csv")
# print(json.dumps(rapport, indent=2))
print("monitor_model() klaar voor gebruik.")


monitor_model() klaar voor gebruik.


# NOTES

- Ze gaan dingen vragen zoals waarom je deze model hebt gekozen, waarom deze parameters, enz. Dus zorg dat je dat goed kan uitleggen.

## Referenties

- Dua, D. & Graff, C. (2019). *Metro Interstate Traffic Volume Data Set*. UCI Machine Learning Repository. https://archive.ics.uci.edu/ml/datasets/Metro+Interstate+Traffic+Volume
- XGBoost Developers. (2024). *XGBoost Documentation*. https://xgboost.readthedocs.io/
- Apache Spark. (2024). *MLlib: Machine Learning Library*. https://spark.apache.org/docs/latest/ml-guide.html
- MLflow. (2024). *MLflow Documentation*. https://mlflow.org/docs/latest/
- FastAPI. (2024). *FastAPI Documentation*. https://fastapi.tiangolo.com/
